In [ ]:
import os
import json
import yaml
import numpy as np
import pandas as pd
import itertools
from collections import Counter
from sklearn.metrics import cohen_kappa_score


In [ ]:
def write_to_json(file, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, 'w') as f:
        json.dump(file, f, indent=2)


def read_json(file_path):
    with open(file_path, "r") as f:
        return json.load(f)
    
def load_sjt(sjt_dir):
    sjt_list = []
    for file in os.listdir(sjt_dir):
        synthetic_sjt = read_json(os.path.join(sjt_dir, file))
        sjt_list += synthetic_sjt

    print(f"{len(sjt_list)} SJTs Loaded")
    return sjt_list

def agg_cohen_kappa(answers):
    cohen_kappa_list = []
    for combo in itertools.combinations(answers, 2):
        cohen_kappa = cohen_kappa_score(combo[0],combo[1])
        cohen_kappa_list.append(cohen_kappa)
        
    return np.round(np.mean(cohen_kappa_list),3).item(),\
        np.round(np.max(cohen_kappa_list),3).item(), \
            np.round(np.min(cohen_kappa_list),3).item()
            
def get_model_name(filename, file_str):
    model_name = filename.replace("_sjt_answers_","").replace(".json","").replace(file_str,"")
    return model_name

In [ ]:
trait_map = {
    "1": "Honesty-Humility",
    "2": "Emotionality",
    "3": "Extraversion",
    "4": "Agreeableness",
    "5": "Conscientiousness",
    "6": "Openness"
}

In [ ]:
def get_trait_distributions(data):
    trait_distributions = []
    true_answers_total = []
    for answer, shuffled_indices in zip(data["answers"], data['config']['answer_index']):
        # assuming one list per persona
        # shuffled_indices = data["config"]["answer_index"]  # list of per-question shuffles

        # Map answers back to their intended trait meaning
        resolved_traits = []
        true_answer_indices = []
        for ans, shuffle in zip(answer, shuffled_indices):
            # ans is the chosen index (1,2,3,...)
            # shuffle is the shuffled order of indices for this question
            true_index = str(shuffle[int(ans)-1] + 1)  # get original index meaning
            if true_index in trait_map:
                resolved_traits.append(trait_map[true_index])
                true_answer_indices.append(true_index)

        trait_counts = Counter(resolved_traits)
        total = sum(trait_counts.values())

        trait_summary = {
            trait: {
                "count": trait_counts[trait],
                "proportion": round(trait_counts[trait] / total, 3) if total > 0 else 0
            }
            for index, trait in trait_map.items()
        }
        trait_distributions.append([trait_summary[key]['proportion'] for key in trait_summary.keys()])
        true_answers_total.append(true_answer_indices)
    return trait_distributions, true_answers_total

In [ ]:
def sjt_trait_summary( persona_dict, trait_map):
    summary = {}
    for persona_id, data in persona_dict.items():
        summary[persona_id] = {}
        trait_distributions, true_answers_total = get_trait_distributions(data)
        
        mean_cohen_kappa, max_cohen_kappa, min_cohen_kappa = agg_cohen_kappa(true_answers_total)
        corr_matrix = pd.DataFrame(trait_distributions).corr()
        
        summary[persona_id] = {
            'mean_cohen_kappa':mean_cohen_kappa,
            'max_cohen_kappa':max_cohen_kappa,
            'min_cohen_kappa':min_cohen_kappa,
            'trait_corr_mat': corr_matrix.values.tolist(),
            'corr_val_cols' : list(trait_map.values())
        }
        
    return summary

In [ ]:
def get_exp_6_metrics(exp_6_data_dir, file_str):
    exp_6_metrics = {}
    for filename in os.listdir(exp_6_data_dir):
        if ".json" in filename and file_str in filename and "gpt" not in filename:
            print(filename)
            model_name = get_model_name(filename, file_str)
            persona_answers = read_json(os.path.join(exp_6_data_dir,filename))
            
            trait_summary = sjt_trait_summary(persona_answers, trait_map)
            
            exp_6_metrics[model_name] = trait_summary
    
    return exp_6_metrics

In [ ]:
exp_6_data_dir = "../experiment_results/reliability_experiments/vllm_experiment_6"

In [ ]:
exp_6_metrics_base_model = get_exp_6_metrics(exp_6_data_dir, "base_model")

In [ ]:
exp_6_metrics_personallm_paper = get_exp_6_metrics(exp_6_data_dir, "personallm_paper")